# k-NN Class Overlap — Measuring the Real Constraint

**Analysis notebook.** Direct follow-up to `label_ambiguity_ceiling.ipynb`, which refuted the
label-noise hypothesis (exact conflict = 0.35% of rows) but left a stronger one standing:

> The limit is **class overlap in CICFlowMeter feature space**. Attack families occupy overlapping
> regions that flow statistics cannot resolve, so every classifier, weighting, pool and decision rule
> inherits the same errors.

The evidence for it was indirect — ambiguity structure predicting model confusion at ρ = 0.70, and
pairs like **Exploits↔Reconnaissance with 451 model confusions and zero exact ambiguity**. Exact
matching cannot measure that, because 99.8% of rows are unique.

## Why k-NN is the right instrument

Replacing *identical* with *nearest* turns a vacuous bound into a real measurement:

| | exact matching | k-NN overlap |
|---|---|---|
| rows it can speak about | 0.35% | 100% |
| ceiling produced | 0.9881 (memorisation) | genuinely achievable |
| Bayes error estimate | none | **Cover–Hart from 1-NN error** |

The Cover–Hart result is the key addition. For a $C$-class problem the asymptotic 1-NN error
satisfies

$$e^\star \;\le\; e_{1\text{NN}} \;\le\; e^\star\left(2 - \tfrac{C}{C-1}e^\star\right)$$

so a measured $e_{1\text{NN}}$ brackets the **Bayes error** — the irreducible error of *any* classifier
on this representation. That is the number the previous notebook could not produce.

## Scope

The **Stage-B sub-problem** (9 attack families, attack rows only) — established as the bottleneck
since the two-stage work, and the gate is at 0.997 recall. Neighbours of **test attack rows** are
sought in **train+val attack rows**, so every measurement is leakage-safe and directly comparable to
the reported test scores.

Distances in **RobustScaler** space, per the project convention for heavy-tailed flow features.

> Runtime ~2–3 min.

## 1. Setup

In [1]:
import numpy as np, pandas as pd, time, warnings
from itertools import combinations
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from scipy.stats import spearmanr

LABELS = {0:"Benign", 1:"Analysis", 2:"Backdoor", 3:"DoS", 4:"Exploits",
          5:"Fuzzers", 6:"Generic", 7:"Reconnaissance", 8:"Shellcode", 9:"Worms"}
RS = 42

X = pd.read_csv("Resource/CIC-UNSW-Data.csv")
y = pd.read_csv("Resource/CIC-UNSW-Label.csv")["Label"]
df = pd.concat([X, y], axis=1).drop_duplicates().reset_index(drop=True)
X, y = df[X.columns], df["Label"]

nun   = X.nunique(); zero = nun[nun <= 1].index.tolist()
corr  = X.drop(columns=zero).corr()
upper = corr.abs().where(np.triu(np.ones(corr.shape, bool), k=1))
red   = [c for c in upper.columns if (upper[c] >= 0.95).any()]
X = X.drop(columns=list(dict.fromkeys(zero + red)))

X_trf, X_test, y_trf, y_test = train_test_split(X, y, test_size=0.2, random_state=RS, stratify=y)

attack_labels = sorted(l for l in y_trf.unique() if l != 0)
fam_to_idx = {f: i for i, f in enumerate(attack_labels)}
NC  = len(attack_labels)
FAM = [LABELS[attack_labels[i]] for i in range(NC)]

Xr = X_trf[y_trf != 0].reset_index(drop=True)                    # reference set
yr = y_trf[y_trf != 0].map(fam_to_idx).reset_index(drop=True).values
Xq = X_test[y_test != 0].reset_index(drop=True)                  # query set
yq = y_test[y_test != 0].map(fam_to_idx).reset_index(drop=True).values

print("reference (train+val attack): %d | query (test attack): %d | features %d"
      % (len(yr), len(yq), X.shape[1]))
print(dict(zip(FAM, np.bincount(yq, minlength=NC).tolist())))

reference (train+val attack): 64936 | query (test attack): 16234 | features 44
{'Analysis': 77, 'Backdoor': 90, 'DoS': 883, 'Exploits': 6182, 'Fuzzers': 5279, 'Generic': 899, 'Reconnaissance': 2406, 'Shellcode': 370, 'Worms': 48}


In [2]:
KMAX = 50
sc = RobustScaler().fit(Xr)
t0 = time.time()
nn = NearestNeighbors(n_neighbors=KMAX, algorithm="brute", n_jobs=-1).fit(sc.transform(Xr))
dist, idx = nn.kneighbors(sc.transform(Xq))
print("neighbours computed in %.0fs" % (time.time() - t0))

YNB = yr[idx]                      # (n_query, KMAX) neighbour labels
print("neighbour label matrix:", YNB.shape)

neighbours computed in 1s
neighbour label matrix: (16234, 50)


## 2. k-NN as an achievable reference

Not a bound — an actual classifier, scored on the same test rows as the ensemble. Two votes:

- **plain** majority — the local Bayes rule under the empirical prior, so it inherits the class
  imbalance and will favour Exploits/Fuzzers.
- **α=0.25 weighted** — each neighbour weighted by the tempered class weights that the weighting study
  established as optimal. This is the fair comparison against the ensemble, which uses the same
  weighting.

Reference: the v3 ensemble's Stage-B macro-F1 on OOF attack rows was **0.6211**.

In [3]:
V3_STAGEB = 0.6211
n_ref = np.bincount(yr, minlength=NC)
wc25  = (n_ref.sum() / n_ref) ** 0.25
wc25 /= wc25.mean()

def vote(k, weights=None):
    nb = YNB[:, :k]
    sc_ = np.zeros((len(nb), NC))
    for c in range(NC):
        m = (nb == c)
        sc_[:, c] = m.sum(1) if weights is None else m.sum(1) * weights[c]
    return sc_.argmax(1)

rows = []
for k in (1, 5, 10, 25, 50):
    for tag, w in (("plain", None), ("alpha=0.25 weighted", wc25)):
        p = vote(k, w)
        rows.append(dict(k=k, vote=tag,
                         macro_F1=round(f1_score(yq, p, average="macro", zero_division=0), 4),
                         macro_P =round(precision_score(yq, p, average="macro", zero_division=0), 4),
                         macro_R =round(recall_score(yq, p, average="macro", zero_division=0), 4),
                         acc     =round(float((p == yq).mean()), 4)))
knn_tab = pd.DataFrame(rows)
print(knn_tab.to_string(index=False))
best = knn_tab.loc[knn_tab.macro_F1.idxmax()]
print("\nbest k-NN macro-F1 : %.4f  (k=%d, %s)" % (best.macro_F1, best.k, best.vote))
print("v3 ensemble Stage-B: %.4f" % V3_STAGEB)
print("gap                : %+.4f" % (V3_STAGEB - best.macro_F1))

 k                vote  macro_F1  macro_P  macro_R    acc
 1               plain    0.4083   0.4129   0.4053 0.6245
 1 alpha=0.25 weighted    0.4083   0.4129   0.4053 0.6245
 5               plain    0.4307   0.5249   0.4006 0.6567
 5 alpha=0.25 weighted    0.4259   0.4107   0.4667 0.6334
10               plain    0.4067   0.5419   0.3725 0.6647
10 alpha=0.25 weighted    0.4458   0.4659   0.4538 0.6541
25               plain    0.3358   0.4848   0.3059 0.6665
25 alpha=0.25 weighted    0.4260   0.4934   0.4242 0.6594
50               plain    0.3153   0.4856   0.2901 0.6603
50 alpha=0.25 weighted    0.3943   0.4591   0.4013 0.6545

best k-NN macro-F1 : 0.4458  (k=10, alpha=0.25 weighted)
v3 ensemble Stage-B: 0.6211
gap                : +0.1753


## 3. Bayes error from the 1-NN rate (Cover–Hart)

Solving $e^\star(2 - \frac{C}{C-1}e^\star) = e_{1\text{NN}}$ for the smaller root gives the lower end of
the bracket; $e_{1\text{NN}}$ itself is the upper end.

This is an **error-rate** statement, so it speaks to accuracy rather than macro-F1 directly — but it
is the first genuine estimate of irreducible error the project has, and it is not vacuous.

In [4]:
e1 = float((YNB[:, 0] != yq).mean())
C  = NC
a, b, c_ = C / (C - 1.0), -2.0, e1                    # a*e^2 + b*e + c = 0
disc = b * b - 4 * a * c_
e_lo = (-b - np.sqrt(disc)) / (2 * a) if disc >= 0 else np.nan

print("1-NN error on test attack rows : %.4f  (accuracy %.4f)" % (e1, 1 - e1))
print("Cover-Hart bracket on Bayes error e*: [%.4f, %.4f]" % (e_lo, e1))
print("  -> irreducible accuracy ceiling roughly %.4f - %.4f" % (1 - e1, 1 - e_lo))
print("\nv3 ensemble Stage-B accuracy on true attack rows, for scale:")
knn_acc = knn_tab.loc[knn_tab.macro_F1.idxmax(), "acc"]
print("  best k-NN accuracy %.4f" % knn_acc)
print("\ncompare: the exact-match bound from the previous notebook put irreducible error at 0.0019")
print("         (vacuous -- attained only by memorising 99.8%% unique rows)")

1-NN error on test attack rows : 0.3755  (accuracy 0.6245)
Cover-Hart bracket on Bayes error e*: [0.2134, 0.3755]
  -> irreducible accuracy ceiling roughly 0.6245 - 0.7866

v3 ensemble Stage-B accuracy on true attack rows, for scale:
  best k-NN accuracy 0.6541

compare: the exact-match bound from the previous notebook put irreducible error at 0.0019
         (vacuous -- attained only by memorising 99.8%% unique rows)


## 4. Per-class purity and the soft ceiling

- **purity** — mean share of a row's $k$ neighbours carrying its own label. Low purity = the class sits
  inside other classes' territory.
- **plurality ceiling** — share of a class's rows where its own label is the *plurality* among
  neighbours. A local-Bayes-rule ceiling on recall, and unlike the exact-match version it is defined
  for every row.

In [5]:
K = 25
nb = YNB[:, :K]
same = (nb == yq[:, None]).mean(1)
plur = vote(K, None)

# achieved per-class F1 from v3 (e2e report; attack families only)
V3_F1 = dict(zip(FAM, [0.455, 0.634, 0.462, 0.773, 0.708, 0.805, 0.657, 0.367, 0.486]))

rows = []
for c in range(NC):
    m = yq == c
    rows.append(dict(family=FAM[c], support=int(m.sum()),
                     purity=round(float(same[m].mean()), 3),
                     plurality_ceiling_R=round(float((plur[m] == c).mean()), 3),
                     achieved_F1=V3_F1[FAM[c]]))
pur = pd.DataFrame(rows).sort_values("purity")
print("k = %d\n" % K)
print(pur.to_string(index=False))

rho, p = spearmanr(pur["purity"], pur["achieved_F1"])
print("\nSpearman rho(purity, achieved F1) = %.3f (p=%.3g)" % (rho, p))
print("-> %s" % ("the families the model does worst on are exactly the least locally pure ones"
                 if rho > 0.6 else "purity only partly explains per-family performance"))

k = 25

        family  support  purity  plurality_ceiling_R  achieved_F1
         Worms       48   0.059                0.000        0.486
     Shellcode      370   0.123                0.062        0.367
      Backdoor       90   0.136                0.122        0.634
           DoS      883   0.174                0.114        0.462
      Analysis       77   0.246                0.039        0.455
       Generic      899   0.359                0.297        0.805
Reconnaissance     2406   0.542                0.507        0.657
       Fuzzers     5279   0.639                0.852        0.708
      Exploits     6182   0.654                0.760        0.773

Spearman rho(purity, achieved F1) = 0.700 (p=0.0358)
-> the families the model does worst on are exactly the least locally pure ones


## 5. Overlap matrix — who sits inside whom

$O[a,b]$ = mean share of class-$a$ rows' neighbours that carry label $b$. The continuous analogue of
the collision-pair table, defined for every row rather than the 0.35% that collide exactly.

In [6]:
O = np.zeros((NC, NC))
for a in range(NC):
    m = yq == a
    for b in range(NC):
        O[a, b] = (nb[m] == b).mean()
print("row = true family, col = share of its %d nearest neighbours\n" % K)
print(pd.DataFrame(O, index=FAM, columns=FAM).round(3).to_string())

print("\nlargest off-diagonal intrusions:")
off = [(FAM[a], FAM[b], O[a, b]) for a in range(NC) for b in range(NC) if a != b]
for a, b, v in sorted(off, key=lambda t: -t[2])[:12]:
    print("  %-15s neighbourhood is %5.1f%% %s" % (a, 100 * v, b))

row = true family, col = share of its 25 nearest neighbours

                Analysis  Backdoor    DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
Analysis           0.246     0.001  0.187     0.514    0.034    0.006           0.009      0.002  0.000
Backdoor           0.002     0.136  0.073     0.320    0.239    0.043           0.160      0.023  0.003
DoS                0.020     0.009  0.174     0.435    0.203    0.059           0.074      0.023  0.003
Exploits           0.007     0.003  0.058     0.654    0.150    0.039           0.072      0.012  0.003
Fuzzers            0.000     0.003  0.029     0.155    0.639    0.029           0.114      0.029  0.003
Generic            0.001     0.003  0.057     0.351    0.180    0.359           0.033      0.012  0.003
Reconnaissance     0.000     0.003  0.022     0.174    0.224    0.013           0.542      0.021  0.002
Shellcode          0.000     0.003  0.053     0.179    0.465    0.028           0.148      0.123  0.001
Wor

## 6. Does overlap explain the model's confusion?

The decisive comparison. Exact-match ambiguity managed ρ = 0.699 while being blind to 99.65% of rows;
if overlap is the real mechanism, this should do better — and in particular should explain the pairs
exact matching missed entirely (Exploits↔Reconnaissance: 451 confusions, 0 collisions).

In [7]:
# v3 test confusion, attack block only (labels 1..9 -> family index 0..8)
CM_FULL = np.array([
 [43546,   51,    2,   18,   73, 1207,  13,   59,  32,  0],
 [    1,   63,    0,    5,    8,    0,   0,    0,   0,  0],
 [    0,    0,   46,    0,   15,   22,   0,    5,   1,  1],
 [    2,   19,    1,  357,  213,  227,  13,   30,  21,  0],
 [    9,   64,    1,  229, 4305, 1213,  58,  246,  52,  5],
 [   32,    2,    2,   18,   74, 4975,  45,   31,  98,  2],
 [    2,    1,    1,   14,   34,  135, 696,    7,   9,  0],
 [    9,    0,    1,   13,  205,  787,   1, 1367,  23,  0],
 [    1,    0,    1,   10,   25,  184,   2,   11, 136,  0],
 [    0,    0,    0,    0,    9,   17,   3,    1,   0, 18]])
CM = CM_FULL[1:, 1:]

AMB = {("Fuzzers","Shellcode"):134, ("Fuzzers","Generic"):87, ("DoS","Fuzzers"):47,
       ("Exploits","Fuzzers"):46, ("DoS","Shellcode"):31, ("DoS","Reconnaissance"):22,
       ("Exploits","Generic"):21, ("Generic","Shellcode"):13, ("Reconnaissance","Shellcode"):12}

recs = []
for a, b in combinations(range(NC), 2):
    key = (FAM[a], FAM[b]); key2 = (FAM[b], FAM[a])
    recs.append(dict(pair="%s|%s" % (FAM[a], FAM[b]),
                     overlap=round(float(O[a, b] + O[b, a]), 4),
                     exact_ambiguity=AMB.get(key, AMB.get(key2, 0)),
                     model_confusion=int(CM[a, b] + CM[b, a])))
comp = pd.DataFrame(recs)

r_ov, p_ov = spearmanr(comp["overlap"], comp["model_confusion"])
r_am, p_am = spearmanr(comp["exact_ambiguity"], comp["model_confusion"])
print("Spearman vs model confusion across %d attack-family pairs:" % len(comp))
print("  k-NN overlap    : rho = %.3f  (p = %.2g)" % (r_ov, p_ov))
print("  exact ambiguity : rho = %.3f  (p = %.2g)" % (r_am, p_am))
print("\ntop 12 by model confusion:")
print(comp.sort_values("model_confusion", ascending=False).head(12).to_string(index=False))

Spearman vs model confusion across 36 attack-family pairs:
  k-NN overlap    : rho = 0.827  (p = 4.9e-10)
  exact ambiguity : rho = 0.538  (p = 0.00071)

top 12 by model confusion:
                    pair  overlap  exact_ambiguity  model_confusion
        Exploits|Fuzzers   0.3051               46             1287
  Fuzzers|Reconnaissance   0.3372                0              818
 Exploits|Reconnaissance   0.2466                0              451
            DoS|Exploits   0.4932                0              442
       Fuzzers|Shellcode   0.4935              134              282
             DoS|Fuzzers   0.2324               47              245
         Fuzzers|Generic   0.2085               87              180
        Exploits|Generic   0.3902               21               92
      Exploits|Shellcode   0.1908                0               77
       Analysis|Exploits   0.5213                0               72
      DoS|Reconnaissance   0.0960               22               43
Rec

## 7. Findings

**The overlap hypothesis is confirmed.** Class overlap in CICFlowMeter feature space explains the
model's errors far better than label noise, and the project now has a real estimate of the
irreducible limit.

| Explanatory variable | Spearman ρ vs model confusion | rows it can speak about |
|---|---|---|
| exact label ambiguity | 0.538 (p = 7×10⁻⁴) | 0.35% |
| **k-NN overlap** | **0.827 (p = 4.9×10⁻¹⁰)** | **100%** |

Decisively, overlap explains the pairs exact matching was blind to:

| Pair | overlap | exact ambiguity | model confusion |
|---|---|---|---|
| Exploits ↔ Reconnaissance | 0.247 | **0** | 451 |
| DoS ↔ Exploits | 0.493 | **0** | 442 |
| Analysis ↔ Exploits | 0.521 | **0** | 72 |

These families never collide exactly and are heavily confused — because they are *inseparable in this
representation*, not mislabelled.

### 1. The first non-vacuous irreducible-error estimate

Cover–Hart on the measured 1-NN error (0.3755) brackets the Bayes error of the 9-class family problem:

$$e^\star \in [0.213,\; 0.376] \quad\Longrightarrow\quad \text{accuracy ceiling} \in [0.625,\; 0.787]$$

For scale, v3's Stage-B accuracy on the 16,234 true attack rows is **0.737** (11,963 correct from the
published confusion matrix) — **inside the bracket, near its upper end**.

The contrast with the previous notebook is the point: exact matching put irreducible error at 0.0019,
a bound attainable only by memorising 99.8% unique rows. This one says roughly **21–38% of attack-family
flows cannot be classified correctly by any method operating on these 44 features**. That is a
publishable statement about the dataset, and it is what the four failed modelling levers were running
into.

### 2. The ensemble is doing real work — k-NN is far behind

| | macro-F1 |
|---|---|
| best k-NN (k=10, α=0.25 weighted) | 0.4458 |
| **v3 ensemble, Stage B** | **0.6211** |
| gap | **+0.1753** |

The prediction that a nearest-neighbour vote might land near the ensemble is **refuted**. The boosted
pool extracts ~17.5 macro-F1 points beyond local neighbourhood structure.

This matters for the thesis narrative. The overlap finding could be read as "the model is trivial and
the data is hard" — it is not. The data is hard *and* the modelling recovers a great deal that a local
classifier cannot. Both halves belong in the write-up.

Note also that k-NN degrades with larger k (plain vote: 0.431 at k=5 → 0.315 at k=50) while accuracy
*rises* (0.657 → 0.660). Larger neighbourhoods drown the rare families entirely — the same
accuracy-vs-macro-F1 tension the whole project has been navigating, visible here in a model with no
hyperparameters to blame.

### 3. Correction: `plurality_ceiling_R` is a misnomer

The column labelled `plurality_ceiling_R` in §4 is **not a ceiling**, and the table shows why: Worms
has a plurality value of **0.000** yet the model achieves F1 0.486; Analysis 0.039 against 0.455;
Shellcode 0.062 against 0.367. The model comfortably exceeds it.

The reason is that plurality voting minimises **error rate**, not macro-F1. A rare class is almost never
the plurality of its own neighbourhood, so the accuracy-optimal local rule assigns it zero recall.
Read the column as *"recall of the accuracy-optimal local rule"* — a **reference point the model
should beat**, and does. The genuine ceiling statement is the Cover–Hart bracket in finding 1.

### 4. Purity ranks the families the way performance does

Spearman ρ(purity, achieved F1) = **0.700 (p = 0.036)**.

| Family | purity (k=25) | achieved F1 |
|---|---|---|
| Worms | 0.059 | 0.486 |
| Shellcode | 0.123 | 0.367 |
| Backdoor | 0.136 | 0.634 |
| DoS | 0.174 | 0.462 |
| Analysis | 0.246 | 0.455 |
| Generic | 0.359 | 0.805 |
| Reconnaissance | 0.542 | 0.657 |
| Fuzzers | 0.639 | 0.708 |
| Exploits | 0.654 | 0.773 |

Only Exploits, Fuzzers and Reconnaissance have neighbourhoods where their own class is a majority.
A Worms flow's 25 nearest neighbours are **5.9% Worms** — it is essentially dissolved in other
families' territory.

### 5. The structure: two attractors swallow everything else

| Family | dominant neighbour class |
|---|---|
| Analysis | **51.4% Exploits** |
| Shellcode | **46.5% Fuzzers** |
| DoS | 43.5% Exploits |
| Worms | 43.4% Exploits |
| Generic | 35.1% Exploits |
| Backdoor | 32.0% Exploits |

**Exploits and Fuzzers are attractors**; the six smaller families are embedded inside them. This is
the same shape as every error pattern the project has found — Fuzzers as false-positive sink (v2,
3,318 FPs), the Fuzzers-centred collision pairs, and the block-structured confusion matrix. One
geometric fact underlies all of them.

### What this settles

The plateau at ~0.63 is a **property of the CICFlowMeter representation**, not a modelling deficiency:

1. It is not label noise (previous notebook: 0.35%).
2. It is class overlap, which predicts model confusion at ρ = 0.83.
3. Roughly 21–38% of attack flows are unclassifiable on these features by any method.
4. The ensemble already sits near the upper end of that bracket, and 17.5 points above a local
   classifier.

Four modelling levers failed because they were all operating downstream of a constraint that lives in
the features.

### Implications for the write-up

- **Reframe the limitation as a measured result.** "Weak rare-class recall" becomes "the rare families
  are geometrically embedded in Exploits/Fuzzers, with purity 0.06–0.25, and we quantify the
  consequence."
- **Future work has a concrete direction**: richer features — payload inspection, host/temporal
  context, session-level aggregation — not better classifiers on the same 44 columns.
- **SHAP should target the overlap pairs** (Exploits↔Reconnaissance, DoS↔Exploits, Fuzzers↔Shellcode)
  rather than the rare classes generically. Explaining *what separates the separable part* of two
  overlapping families is a sharper contribution than a global feature-importance plot.

### Limitations

- Euclidean distance in RobustScaler space is one geometry; a learned metric could show less overlap.
  The finding is about *this* representation under *this* metric.
- Cover–Hart is asymptotic. With 64,936 reference points in 44 dimensions the asymptotic regime is not
  guaranteed, so the bracket is indicative rather than exact — and it is wide (0.213–0.376).
- The bracket constrains **error rate**, not macro-F1. The rare-class story rests on the purity table.
